# Impulse — A2D2 Object Detection → Time-Series Channels

Third notebook of the A2D2 demo. It runs a **pretrained torchvision COCO object detector
(BSD-3-Clause)** over the camera frames already extracted to the Unity Catalog volume,
**distributed across Spark workers** (`mapInPandas`), and turns detections into **numerical
time-series channels** appended to the silver layer (first-class TSAL channels):

- **count** channels — e.g. `detected_pedestrians` = number of pedestrians per frame;
- **distance** channels (optional, `estimate_distance`) — e.g. `pedestrian_nearest_distance_m`
  = distance in metres to the nearest pedestrian, via **Depth Anything V2 metric depth
  (Apache-2.0)** sampled inside each detection box.

Everything stays in Spark: the UDF output is persisted to a staging table and the silver
tables are built from it with Spark (no driver-side `toPandas` collect).

## Prerequisite
Run `a2d2_ingestion.ipynb` first (with `download_images=true`). torchvision/Depth-Anything
weights are BSD-3/Apache-2.0; the A2D2 data and weights stay on your own volume — commit the
notebook with outputs cleared.

In [ ]:
%pip install -U transformers
dbutils.library.restartPython()

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text("table_prefix", "a2d2", "Table Prefix")
dbutils.widgets.text("volume", "a2d2_raw", "UC Volume")
dbutils.widgets.dropdown("detect_model", "fasterrcnn_resnet50_fpn",
    ["fasterrcnn_resnet50_fpn", "ssdlite320_mobilenet_v3_large", "fcos_resnet50_fpn"], "Detection model")
dbutils.widgets.text("detect_score_threshold", "0.5", "Score threshold")
dbutils.widgets.text("frames_per_second", "1", "Frames/sec to detect (0 = all frames)")
dbutils.widgets.text("ahead_center_frac", "0.34", "Center-ahead band as fraction of image width")
dbutils.widgets.text("detect_repartitions", "16", "Inference partitions")
dbutils.widgets.dropdown("estimate_distance", "true", ["true", "false"], "Estimate distance (depth)")
dbutils.widgets.text("depth_model", "depth-anything/Depth-Anything-V2-Metric-Outdoor-Small-hf", "Depth model (HF)")

In [ ]:
import sys, os

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")
VOLUME = dbutils.widgets.get("volume")
MODEL_NAME = dbutils.widgets.get("detect_model")
SCORE_THRESHOLD = float(dbutils.widgets.get("detect_score_threshold") or "0.5")
FRAMES_PER_SECOND = float(dbutils.widgets.get("frames_per_second") or "0")
AHEAD_CENTER_FRAC = float(dbutils.widgets.get("ahead_center_frac") or "0.34")
REPARTITIONS = int(dbutils.widgets.get("detect_repartitions") or "16")
ESTIMATE_DISTANCE = dbutils.widgets.get("estimate_distance") == "true"
DEPTH_MODEL = dbutils.widgets.get("depth_model")

if not (CATALOG and SCHEMA and TABLE_PREFIX and VOLUME):
    raise ValueError("Please set catalog, schema, table_prefix and volume widgets above.")

nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
REPO_ROOT = "/Workspace" + "/".join(nb_path.split("/")[:-3])
src_dir = os.path.join(REPO_ROOT, "src")
if os.path.isdir(src_dir):
    sys.path.insert(0, src_dir)

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"
CONTAINER_ID = 1
vol_root = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
STAGING = f"{pfx}_detect_staging"

# Count channels: curated COCO classes -> per-frame detection count (name-keyed, gap-safe).
COUNT_CLASS_MAP = {
    "detected_pedestrians":    {"person"},
    "detected_cyclists":       {"bicycle"},
    "detected_motorcycles":    {"motorcycle"},
    "detected_cars":           {"car"},
    "detected_buses":          {"bus"},
    "detected_trucks":         {"truck"},
    "detected_vehicles":       {"car", "bus", "truck", "motorcycle"},
    "detected_traffic_lights": {"traffic light"},
    "detected_stop_signs":     {"stop sign"},
}
# Distance channels: nearest metric distance (m) to that class per frame (NaN if absent).
DISTANCE_CLASS_MAP = {
    "pedestrian_nearest_distance_m": {"person"},
    "cyclist_nearest_distance_m":    {"bicycle"},
    "vehicle_nearest_distance_m":    {"car", "bus", "truck", "motorcycle"},
}
# "Center-ahead" variants: same classes, but counted/measured only for detections whose
# bounding box is in the central forward band (~ego path) -> in-lane lead object, less noise
# from oncoming/parked/cross-traffic. Mirrors the count + distance maps for ALL classes.
AHEAD_COUNT_CLASS_MAP = {n.replace("detected_", "ahead_"): cls for n, cls in COUNT_CLASS_MAP.items()}
AHEAD_DISTANCE_CLASS_MAP = {n.replace("_nearest_distance_m", "_ahead_nearest_distance_m"): cls
                            for n, cls in DISTANCE_CLASS_MAP.items()}

COUNT_CHANNELS = list(COUNT_CLASS_MAP.keys())
DISTANCE_CHANNELS = list(DISTANCE_CLASS_MAP.keys()) if ESTIMATE_DISTANCE else []
AHEAD_COUNT_CHANNELS = list(AHEAD_COUNT_CLASS_MAP.keys())
AHEAD_DISTANCE_CHANNELS = list(AHEAD_DISTANCE_CLASS_MAP.keys()) if ESTIMATE_DISTANCE else []
# Order keeps existing ids stable (counts 100-108, distances 109-111) and appends ahead channels.
ALL_CHANNELS = COUNT_CHANNELS + DISTANCE_CHANNELS + AHEAD_COUNT_CHANNELS + AHEAD_DISTANCE_CHANNELS
DETECT_ID_BASE = 100  # detection ids 100.. (never collide with bus ids 1..22)
channel_id_for = {name: DETECT_ID_BASE + i for i, name in enumerate(ALL_CHANNELS)}
UNIT_OF = {**{c: "count" for c in COUNT_CHANNELS}, **{c: "m" for c in DISTANCE_CHANNELS},
           **{c: "count" for c in AHEAD_COUNT_CHANNELS}, **{c: "m" for c in AHEAD_DISTANCE_CHANNELS}}
print(f"model={MODEL_NAME} thr={SCORE_THRESHOLD} estimate_distance={ESTIMATE_DISTANCE}")
print("channels:", ALL_CHANNELS)

## Channels produced

**Counts** (unit `count`): `detected_pedestrians`=person, `detected_cyclists`=bicycle,
`detected_motorcycles`, `detected_cars`, `detected_buses`, `detected_trucks`,
`detected_vehicles`=car+bus+truck+motorcycle, `detected_traffic_lights`, `detected_stop_signs`.

**Distances** (unit `m`, when `estimate_distance=true`): `pedestrian_nearest_distance_m`,
`cyclist_nearest_distance_m`, `vehicle_nearest_distance_m` — the nearest object of that class
per frame, NaN when none present.

**Center-ahead** variants (same classes, counted/measured only for detections whose bounding-box
center is in the central forward band `[0.5±ahead_center_frac/2]·W` ≈ the ego path): `ahead_*`
counts (`ahead_pedestrians` … `ahead_vehicles`, `ahead_traffic_lights`, `ahead_stop_signs`) and
`*_ahead_nearest_distance_m` distances. These approximate the **in-lane lead object**, filtering
out oncoming / parked / cross-traffic (e.g. `vehicle_ahead_nearest_distance_m` for rear-end risk).

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

frames_df = (spark.read.table(f"{pfx}_camera_frames")
             .select("container_id", "frame_id", "cam_tstamp_us", "png_path")
             .where("png_path IS NOT NULL"))
total_frames = frames_df.count()
if total_frames == 0:
    raise RuntimeError(f"{pfx}_camera_frames is empty - run a2d2_ingestion with download_images=true first.")

# Optional temporal downsampling: keep the earliest frame in each 1/fps-second bucket
# (robust to frame-rate jitter/gaps). frames_per_second=0 -> score every frame.
if FRAMES_PER_SECOND and FRAMES_PER_SECOND > 0:
    bucket_us = int(round(1e6 / FRAMES_PER_SECOND))
    # Partition by container_id too, so each drive is subsampled independently.
    _w = Window.partitionBy(F.col("container_id"), (F.col("cam_tstamp_us") / F.lit(bucket_us)).cast("long")).orderBy("cam_tstamp_us")
    frames_df = frames_df.withColumn("_rn", F.row_number().over(_w)).where("_rn = 1").drop("_rn")
n_frames = frames_df.count()
rate = f"~{FRAMES_PER_SECOND} fps" if FRAMES_PER_SECOND and FRAMES_PER_SECOND > 0 else "all frames"
print(f"{n_frames} of {total_frames} camera frames selected ({rate}) to score with {MODEL_NAME}"
      + (f" + depth ({DEPTH_MODEL})" if ESTIMATE_DISTANCE else ""))

In [ ]:
import torch
from torchvision.models import detection as D

MODELS = {
    "fasterrcnn_resnet50_fpn":       (D.fasterrcnn_resnet50_fpn,       D.FasterRCNN_ResNet50_FPN_Weights),
    "ssdlite320_mobilenet_v3_large": (D.ssdlite320_mobilenet_v3_large, D.SSDLite320_MobileNet_V3_Large_Weights),
    "fcos_resnet50_fpn":             (D.fcos_resnet50_fpn,             D.FCOS_ResNet50_FPN_Weights),
}
ctor, wenum = MODELS[MODEL_NAME]
weights = wenum.DEFAULT
COCO_CATEGORIES = list(weights.meta["categories"])

# Detector weights -> volume (driver downloads once; workers load the state_dict from FUSE).
models_dir = f"{vol_root}/models"
os.makedirs(models_dir, exist_ok=True)
weights_path = f"{models_dir}/{MODEL_NAME}.pth"
if not os.path.exists(weights_path):
    print("downloading detector weights on driver ...")
    _m = ctor(weights=weights)
    torch.save(_m.state_dict(), weights_path)
    del _m
print("detector weights:", weights_path)

# Depth model -> volume (driver downloads once via transformers; workers load from FUSE dir).
DEPTH_DIR = None
if ESTIMATE_DISTANCE:
    from transformers import AutoModelForDepthEstimation, AutoImageProcessor
    DEPTH_DIR = f"{models_dir}/" + DEPTH_MODEL.replace("/", "__")
    if not os.path.exists(os.path.join(DEPTH_DIR, "config.json")):
        print("downloading depth model on driver ...")
        _dm = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL)
        _dp = AutoImageProcessor.from_pretrained(DEPTH_MODEL)
        _dm.save_pretrained(DEPTH_DIR)
        _dp.save_pretrained(DEPTH_DIR)
        del _dm, _dp
    print("depth model:", DEPTH_DIR)

# Resolve channel -> set of COCO indices on the driver (captured by the UDF closure;
# serverless forbids spark.sparkContext.broadcast, and these are tiny).
name_to_idx = {c: i for i, c in enumerate(COCO_CATEGORIES)}
def _idx_map(class_map):
    return {ch: sorted({name_to_idx[n] for n in names if n in name_to_idx})
            for ch, names in class_map.items()}
COUNT_IDX = _idx_map(COUNT_CLASS_MAP)
DIST_IDX = _idx_map(DISTANCE_CLASS_MAP) if ESTIMATE_DISTANCE else {}
AHEAD_COUNT_IDX = _idx_map(AHEAD_COUNT_CLASS_MAP)
AHEAD_DIST_IDX = _idx_map(AHEAD_DISTANCE_CLASS_MAP) if ESTIMATE_DISTANCE else {}
print("count idx:", COUNT_IDX)

## Distributed inference (`mapInPandas`) → staging table

Each partition loads the detector (and depth model, if enabled) once from the cached
weights, scores its frames on CPU, and emits long-form rows `(container_id, cam_tstamp_us,
channel_name, value)` — `value` is a count for count channels and a distance (m) for
distance channels (NaN if the class is absent). The UDF output is **persisted to a staging
table** so the rest of the pipeline stays in Spark.

In [ ]:
import pyspark.sql.types as T
import re

# Bounding box struct: pixel coordinates + detection confidence score + class name
BOX_STRUCT = T.StructType([
    T.StructField("x1", T.DoubleType()),
    T.StructField("y1", T.DoubleType()),
    T.StructField("x2", T.DoubleType()),
    T.StructField("y2", T.DoubleType()),
    T.StructField("score", T.DoubleType()),
    T.StructField("class", T.StringType()),
])

DETECT_OUT_SCHEMA = T.StructType([
    T.StructField("container_id",  T.LongType(),   False),
    T.StructField("cam_tstamp_us", T.LongType(),   False),
    T.StructField("channel_name",  T.StringType(), False),
    T.StructField("sensor_name",   T.StringType(), False),
    # Nullable: distance channels emit NaN when the class is absent in a frame, and PySpark's
    # pandas->Arrow conversion turns float NaN into a null -> a non-nullable field would crash.
    T.StructField("value",         T.DoubleType(), True),
    # Bounding boxes for detections of this channel's class(es), with confidence scores.
    # Enables downstream object tracking and richer analysis.
    T.StructField("boxes",         T.ArrayType(BOX_STRUCT), True),
])

# Regex to extract sensor name from path (e.g., "cam_front_center" from ".../camera/cam_front_center/...")
SENSOR_PATTERN = re.compile(r"/camera/(cam_[^/]+)/")

def detect_partition(iterator):
    import torch, numpy as np
    from torchvision.models import detection as _D
    import torchvision.transforms.functional as _TF
    from PIL import Image
    from collections import Counter
    import pandas as pd
    import re

    ctors = {
        "fasterrcnn_resnet50_fpn":       _D.fasterrcnn_resnet50_fpn,
        "ssdlite320_mobilenet_v3_large": _D.ssdlite320_mobilenet_v3_large,
        "fcos_resnet50_fpn":             _D.fcos_resnet50_fpn,
    }
    # Load once per partition (architecture only -> NO download; weights from the FUSE volume).
    model = ctors[MODEL_NAME](weights=None, weights_backbone=None)
    model.load_state_dict(torch.load(weights_path, map_location="cpu"))
    model.eval()
    torch.set_num_threads(1)
    thr = SCORE_THRESHOLD
    use_depth = ESTIMATE_DISTANCE and len(DIST_IDX) > 0
    depth_model = depth_proc = None
    if use_depth:
        from transformers import AutoModelForDepthEstimation, AutoImageProcessor
        depth_model = AutoModelForDepthEstimation.from_pretrained(DEPTH_DIR)
        depth_model.eval()
        depth_proc = AutoImageProcessor.from_pretrained(DEPTH_DIR)

    for pdf in iterator:
        rows = []
        for _, r in pdf.iterrows():
            png_path = r["png_path"]
            try:
                img = Image.open(png_path).convert("RGB")
            except Exception:
                continue
            
            # Extract sensor name from path (e.g., "cam_front_center")
            sensor_match = SENSOR_PATTERN.search(png_path)
            sensor_name = sensor_match.group(1) if sensor_match else "unknown"
            
            W, H = img.size
            x = _TF.to_tensor(img)
            with torch.no_grad():
                pred = model([x])[0]
            keep = pred["scores"] >= thr
            scores = pred["scores"][keep].tolist()  # Keep detection confidence scores
            labels = pred["labels"][keep].tolist()
            boxes = pred["boxes"][keep].tolist()
            cid, ts = int(r["container_id"]), int(r["cam_tstamp_us"])
            # center-ahead mask: box horizontal center within the central band (~ego path)
            lo, hi = (0.5 - AHEAD_CENTER_FRAC / 2) * W, (0.5 + AHEAD_CENTER_FRAC / 2) * W
            ahead = [lo <= (b[0] + b[2]) / 2 <= hi for b in boxes]

            def boxes_for_classes(idxs, mask=None):
                """Return list of {x1, y1, x2, y2, score, class} for detections matching idxs."""
                result = []
                for k, (lab, box, sc) in enumerate(zip(labels, boxes, scores)):
                    if lab in idxs and (mask is None or mask[k]):
                        result.append({"x1": box[0], "y1": box[1],
                                      "x2": box[2], "y2": box[3], "score": sc,
                                      "class": COCO_CATEGORIES[lab]})
                return result if result else None

            cnt = Counter(labels)
            for ch, idxs in COUNT_IDX.items():
                ch_boxes = boxes_for_classes(idxs)
                rows.append((cid, ts, ch, sensor_name, float(sum(cnt.get(i, 0) for i in idxs)), ch_boxes))
            for ch, idxs in AHEAD_COUNT_IDX.items():
                ch_boxes = boxes_for_classes(idxs, ahead)
                rows.append((cid, ts, ch, sensor_name, float(sum(1 for lab, a in zip(labels, ahead) if a and lab in idxs)), ch_boxes))
            if use_depth:
                with torch.no_grad():
                    di = depth_proc(images=img, return_tensors="pt")
                    pd_depth = depth_model(**di).predicted_depth      # (1, h, w) metres
                depth = torch.nn.functional.interpolate(
                    pd_depth.unsqueeze(1), size=(H, W), mode="bilinear", align_corners=False)[0, 0].cpu().numpy()
                def _nearest(idxs, mask=None):
                    ds = []
                    for k, (lab, box) in enumerate(zip(labels, boxes)):
                        if lab in idxs and (mask is None or mask[k]):
                            x1, y1, x2, y2 = [int(round(v)) for v in box]
                            x1, y1 = max(0, x1), max(0, y1)
                            x2, y2 = min(W, x2), min(H, y2)
                            if x2 > x1 and y2 > y1:
                                patch = depth[y1:y2, x1:x2]
                                if patch.size:
                                    ds.append(float(np.median(patch)))
                    return float(min(ds)) if ds else float("nan")
                for ch, idxs in DIST_IDX.items():
                    ch_boxes = boxes_for_classes(idxs)
                    rows.append((cid, ts, ch, sensor_name, _nearest(idxs), ch_boxes))
                for ch, idxs in AHEAD_DIST_IDX.items():
                    ch_boxes = boxes_for_classes(idxs, ahead)
                    rows.append((cid, ts, ch, sensor_name, _nearest(idxs, ahead), ch_boxes))
        if rows:
            yield pd.DataFrame(rows, columns=["container_id", "cam_tstamp_us", "channel_name", "sensor_name", "value", "boxes"])

# Run inference and PERSIST to a staging table (materialises once, in Spark - no toPandas).
(frames_df.repartition(REPARTITIONS)
 .mapInPandas(detect_partition, schema=DETECT_OUT_SCHEMA)
 .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(STAGING))
print("detections staged to", STAGING, "rows:", spark.read.table(STAGING).count())

## Build silver channels from the staging table (Spark)

`channels`, `channel_tags` and `channel_metrics` are derived from the staging table entirely
in Spark: a channel-mapping DataFrame supplies `channel_id`/`unit`; metrics are computed
distributed via `groupBy(channel_name).applyInPandas(...)` (duration-weighted, NaN-aware).

In [ ]:
import pyspark.sql.functions as F
from impulse_query_engine import schema as S

det = spark.read.table(STAGING)
SOURCE = f"torchvision:{MODEL_NAME}" + (f"+depth:{DEPTH_MODEL}" if ESTIMATE_DISTANCE else "")

# channel_name -> (channel_id, unit) mapping DataFrame
map_df = spark.createDataFrame(
    [(name, int(channel_id_for[name]), UNIT_OF[name]) for name in ALL_CHANNELS],
    "channel_name string, channel_id int, unit string",
)

# CHANNELS (RAW points)
channels_df = (det.join(map_df, "channel_name")
    .select(F.col("container_id").cast("long"),
            F.col("channel_id").cast("int"),
            F.col("cam_tstamp_us").cast("long").alias("timestamp"),
            F.col("value").cast("double")))

# CHANNEL_TAGS (channel_name / unit / source) — one set per container present in staging
containers_df = det.select("container_id").distinct()
def _tag(key_col, val_col):
    return (map_df.crossJoin(containers_df)
            .select(F.col("container_id").cast("long"),
                    F.col("channel_id"), F.lit(key_col).alias("key"), val_col.alias("value")))
channel_tags_df = (_tag("channel_name", F.col("channel_name"))
                   .unionByName(_tag("unit", F.col("unit")))
                   .unionByName(_tag("source", F.lit(SOURCE))))

# CHANNEL_METRICS via applyInPandas (distributed, duration-weighted, NaN-aware; pure numpy)
METRICS_SCHEMA = T.StructType([
    T.StructField("container_id", T.LongType()),
    T.StructField("channel_name", T.StringType()),
    T.StructField("value_type", T.StringType()),
    T.StructField("sample_count", T.IntegerType()),
    T.StructField("nan_ratio", T.FloatType()),
    T.StructField("begin_s", T.FloatType()),
    T.StructField("end_s", T.FloatType()),
    T.StructField("duration_ms", T.IntegerType()),
    T.StructField("original_sample_count", T.IntegerType()),
    T.StructField("original_sr", T.FloatType()),
    T.StructField("min", T.FloatType()),
    T.StructField("max", T.FloatType()),
    T.StructField("mean", T.FloatType()),
    T.StructField("std", T.FloatType()),
    T.StructField("pz1", T.FloatType()),
    T.StructField("pz10", T.FloatType()),
    T.StructField("pz90", T.FloatType()),
    T.StructField("pz99", T.FloatType()),
])

def compute_metrics(pdf):
    import numpy as np, pandas as pd
    name = pdf["channel_name"].iloc[0]
    cid = int(pdf["container_id"].iloc[0])
    pdf = pdf.sort_values("cam_tstamp_us")
    times = pdf["cam_tstamp_us"].to_numpy(np.float64)
    values = pdf["value"].to_numpy(np.float64)
    n = int(len(values))
    dur = (np.append(np.diff(times), 0.0) / 1e6) if n else np.array([])   # seconds; last sample 0
    valid = ~np.isnan(values)
    tot = dur.sum()
    nan_ratio = float(dur[~valid].sum() / tot) if tot > 0 else 0.0
    begin_s = float(times[0] / 1e6) if n else 0.0
    end_s = float(times[-1] / 1e6) if n else 0.0
    duration_ms = int((times[-1] - times[0]) / 1e3) if n else 0
    original_sr = float(np.mean(np.diff(times)) / 1e6) if n > 1 else 0.0
    if valid.any():
        vv, dw = values[valid], dur[valid]
        wsum = dw.sum()
        mn, mx = float(np.min(vv)), float(np.max(vv))
        mean = float(np.sum(vv * dw) / wsum) if wsum > 0 else float(np.mean(vv))
        std = float(np.sqrt(np.sum(dw * (vv - mean) ** 2) / wsum)) if wsum > 0 else 0.0
        order = np.argsort(vv); sv, sw = vv[order], dw[order]
        cw = np.cumsum(sw)
        pz = [float(np.interp(q, cw / cw[-1], sv)) for q in (0.01, 0.10, 0.90, 0.99)] if cw[-1] > 0 else [mn, mn, mx, mx]
    else:
        mn = mx = mean = std = float("nan"); pz = [float("nan")] * 4
    return pd.DataFrame([dict(container_id=cid, channel_name=name, value_type="DOUBLE", sample_count=n,
        nan_ratio=nan_ratio, begin_s=begin_s, end_s=end_s, duration_ms=duration_ms,
        original_sample_count=n, original_sr=original_sr, min=mn, max=mx, mean=mean, std=std,
        pz1=pz[0], pz10=pz[1], pz90=pz[2], pz99=pz[3])])

# Group by (container_id, channel_name) so each drive gets its own per-channel metrics.
channel_metrics_df = (det.groupBy("container_id", "channel_name").applyInPandas(compute_metrics, METRICS_SCHEMA)
    .join(map_df.select("channel_name", "channel_id"), "channel_name")
    .select(F.col("container_id").cast("long"), "channel_id", "value_type",
            "sample_count", "nan_ratio", "begin_s", "end_s", "duration_ms", "original_sample_count",
            "original_sr", "min", "max", "mean", "std", "pz1", "pz10", "pz90", "pz99"))

In [ ]:
# Idempotent append: detection runs over ALL containers in one go, so clear the whole
# detection id range across every container, then append.
LO, HI = DETECT_ID_BASE, DETECT_ID_BASE + 100
for tbl, df in [("channels", channels_df), ("channel_tags", channel_tags_df), ("channel_metrics", channel_metrics_df)]:
    full = f"{pfx}_{tbl}"
    spark.sql(f"DELETE FROM {full} WHERE channel_id >= {LO} AND channel_id < {HI}")
    df.write.mode("append").saveAsTable(full)

# Refresh num_channels per container (bus + detection channels now present).
for r in (spark.read.table(f"{pfx}_channel_tags").where("key = 'channel_name'")
          .groupBy("container_id").agg(F.countDistinct("channel_id").alias("n")).collect()):
    spark.sql(f"UPDATE {pfx}_container_metrics SET num_channels = {int(r['n'])} "
              f"WHERE container_id = {int(r['container_id'])}")
conts = [int(r["container_id"]) for r in spark.read.table(f"{pfx}_channel_tags").select("container_id").distinct().collect()]
print(f"Appended {len(ALL_CHANNELS)} detection channels per container (ids {sorted(channel_id_for.values())}) "
      f"for containers {sorted(conts)}.")

## Validation — detection channels are first-class TSAL channels

In [ ]:
from databricks.sdk import WorkspaceClient
from impulse_reporting.core.report import Report

config = {
    "source": {
        "container_metrics_table": f"{pfx}_container_metrics",
        "container_tags_table": f"{pfx}_container_tags",
        "channel_metrics_table": f"{pfx}_channel_metrics",
        "channel_tags_table": f"{pfx}_channel_tags",
        "channels_uri": f"{pfx}_channels",
    },
    "unity_sink": {"catalog": CATALOG, "schema": SCHEMA, "table_prefix": TABLE_PREFIX},
    "query_engine": {"solver": "DeltaSolver", "data_type": "RAW"},
    "measurement_dimensions": ["container_id"],
}
report = Report(name="a2d2_detect", spark=spark, workspace_client=WorkspaceClient(), config=config)
db = report.get_db()
_ = db.query.channel(channel_name="detected_pedestrians")
if ESTIMATE_DISTANCE:
    _ = db.query.channel(channel_name="pedestrian_nearest_distance_m")
print("OK: detection channels resolve as TSAL channels.")

import pyspark.sql.functions as F
print("Per-(container, channel) summary (avg value, non-null count):")
display(spark.read.table(STAGING).groupBy("container_id", "channel_name")
        .agg(F.round(F.avg("value"), 3).alias("avg_value"),
             F.count(F.when(~F.isnan("value"), 1)).alias("non_null"))
        .orderBy("container_id", "channel_name"))
id_list = ",".join(str(channel_id_for[n]) for n in ALL_CHANNELS)
display(spark.read.table(f"{pfx}_channel_metrics").where(f"channel_id IN ({id_list})")
        .select("container_id", "channel_id", "sample_count", "nan_ratio", "min", "max", "mean")
        .orderBy("container_id", "channel_id"))